In [ ]:
from dotenv import load_dotenv
import os

In [ ]:
from langchain_groq import ChatGroq
groq_llm = ChatGroq(model="llama-3.1-8b-instant")

In [ ]:
llm_response= groq_llm.invoke("Hello, who are you?")
llm_response.content

In [ ]:
from typing_extensions import TypedDict, Annotated
import operator

In [ ]:
from langchain_core.messages import AnyMessage , HumanMessage, AIMessage

In [ ]:
# Defining State
class GraphState(TypedDict):
    messages : Annotated[list[AnyMessage], operator.add]

In [ ]:
# First Node

def llm_call(state : GraphState) -> dict:
    """Call the LLM using conversation messages and append AI Responses"""
    response = groq_llm.invoke(state["messages"])
    return {
        "messages" :[response]
        #messages" :state['messages'] + [response]
    }

In [ ]:
# Second Node
def token_counter(state:GraphState)->dict:
    """Count tokens(simple word count) in the last AI Message"""
    last_msg = state['messages'][-1]
    text = last_msg.content
    token_number = len(text.split())
    Summary = f"Total token number in the generated answer (word count) is {token_number}"

    return {
       "messages" : [AIMessage(content=Summary)]
       # "messages" : state['messages']
    }

Finally, We have to Orchestrate the workflow

In [ ]:
# Orchestration -> step by step executing , sequential manner
from langgraph.graph import StateGraph

builder = StateGraph(GraphState)

# adding Nodes

builder.add_node("llm_call", llm_call)
builder.add_node("token_counter",token_counter)

In [ ]:
# Add Edges
builder.set_entry_point("llm_call")
builder.add_edge("llm_call","token_counter")
builder.set_finish_point("token_counter")

app=builder.compile()

visualize the Graph

In [ ]:
app.get_graph()

In [ ]:
from IPython.display import Image,display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# Calling entire flow / Graph -> use invoke,astream

In [ ]:
result = app.invoke({
    "messages" : [HumanMessage(content="Hi , This is Venkatesh,say hello in detail")]
})

In [ ]:
result

In [ ]:
for M in result['messages']:
    print(type(M).__name__,":", M.content)

In [ ]:
result = app.invoke({
    "messages" : [HumanMessage(content="Good to hear you ! and Bye Now !")] 
})

In [ ]:
for M in result['messages']:
    print(type(M).__name__,":", M.content)

In [ ]:
result

- If you use [response], the old user message "Hello" is discarded — only the AI’s reply remains.
- If you want to keep the conversation history, you should append instead

return {
    "messages": state["messages"] + [response]
}


Tool Calling 

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [ ]:
api_wrapper = WikipediaAPIWrapper(top_k_results=5, doc_content_chars_max=500)

In [ ]:
wiki_tool =WikipediaQueryRun(api_wrapper=api_wrapper)

In [ ]:
wiki_tool.run({"query":"Transformers in Large language models"})

In [ ]:
import os
from langchain_community.tools.tavily_search import TavilySearchResults

In [ ]:
tool = TavilySearchResults()
tavily_tool = tool.invoke({"query":"How is the future of LLM?"})
tavily_tool

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults
search = DuckDuckGoSearchResults()

In [ ]:
duckduck_tool = search.invoke("what's latest news on LLM?")

In [ ]:
# BingSearch , googleserpai API

In [ ]:
from langchain_community.tools import YouTubeSearchTool
youtube_tool = YouTubeSearchTool()

In [ ]:
youtube_tool.name

In [ ]:
youtube_tool.run("LLM")

In [ ]:
# A Simple Method , can not call run or Invoke method
def multiply(a:int , b:int)-> int:
    return a *b

multiply(10,20)

In [ ]:
# Use @tool decorator
from langchain.tools import tool

@tool
def multiply(y:int , z:int)->any:

    """Y : takes the integer input
       z : take the integer input
       return the multiplied value anytype"""
    
    return y*z

In [ ]:
multiply.invoke({"y":50,"z":40})

In [ ]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

In [ ]:
@tool
def length_word(word:str)-> int:
    """it is a tool to count the length of the words"""
    return len(word)

In [ ]:
length_word.invoke("hello large language models  and tool calling with Decorator")

In [ ]:
length_word.args

In [ ]:
import yfinance as yf

In [ ]:
@tool
def get_Stock_price(ticker:str)-> str:

    """this is tool to get the stock price using yfinance"""
    try:

        stock = yf.Ticker(ticker) # ticker name of the stock

        # Get last 1 day historical data

        data = stock.history(period ="1d")

        if data.empty:
            return f"No data found for ticker {ticker} . please check the symbol"
        
        latest_close = data["Close"].iloc[-1]

        currency = stock.info.get("currency","")

        symbol_map = {
            "INR": "₹",
            "USD": "$",
            "EUR": "€",
            "GBP": "£",
            "JPY": "¥",
            "CNY": "¥",
            "AUD": "A$",
            "CAD": "C$",
            "CHF": "CHF",
            "KRW": "₩",
            "RUB": "₽",
            "TRY": "₺",
            "BTC": "₿"
        }

        symbol =symbol_map.get(currency,"")
        currency_text = currency if currency else ""

        if symbol:
            return f"the last closing price of {ticker.upper()} was {symbol}{latest_close:.2f}"
        else:
            return f"the last closing price of {ticker.upper()} was {latest_close:.2f}{currency_text}"


    except Exception as e:
        return f"An Error occured while fetching stock data : {str(e)}"

In [ ]:
get_Stock_price.invoke("TCS.NS")

In [ ]:
get_Stock_price.invoke("CTS")

In [ ]:
get_Stock_price.invoke("AAPL")

In [ ]:
get_Stock_price.invoke("TSLA")

In [ ]:
get_Stock_price.invoke("HDFC")

How LLM's will decide which tool needs to be call

In [ ]:
# Define List of all tools - in List
tools =[get_Stock_price,multiply,length_word,wiki_tool]

In [ ]:
# bind with the LLM
llm_with_tools = groq_llm.bind_tools(tools)

In [ ]:
result = llm_with_tools.invoke("what is the stock price of TCS.NS?")

In [ ]:
result

In [ ]:
result.tool_calls

In [ ]:
result.content

In [ ]:
word_length_tool= llm_with_tools.invoke("How many words in the sentence, 'Hello world , this is a test sentence' ")
word_length_tool

In [ ]:
word_length_tool.tool_calls

In [ ]:
multiply_tool= llm_with_tools.invoke("what is the multiplication of 2 and 3?")
multiply_tool

In [ ]:
multiply_tool.tool_calls

In [ ]:
# Normal scenario, with out tool, LLM giving answer
normal_question= llm_with_tools.invoke("Hi , How are you?")

In [ ]:
normal_question

In [ ]:
normal_question.tool_calls

In [ ]:
tool_calling_question= llm_with_tools.invoke("what was the latest india union budget of India 2026? ")

In [ ]:
tool_calling_question.tool_calls

In [ ]:
tool_calling_question.content

In [ ]:
from langchain_core.messages import HumanMessage,SystemMessage
from langgraph.graph import MessagesState,StateGraph,END,START
from langgraph.prebuilt import ToolNode,tools_condition

In [ ]:
System_prompt = SystemMessage(content="your a helpful assistant that can use tools to answer questions.")

In [ ]:
def function_1(state: MessagesState):
    
    user_question = state['messages']

    input_question = [System_prompt]+user_question

    response = llm_with_tools.invoke(input_question)
    return{
        "messages":[response]
    }


In [ ]:
function_2 = ToolNode(tools=tools)

In [ ]:
workflow= StateGraph(MessagesState)

workflow .add_node("llm", function_1)
workflow.add_node("tools",function_2)

In [ ]:
workflow.add_edge(START,"llm")

In [ ]:
workflow.add_conditional_edges("llm",tools_condition,)

In [ ]:
workflow.add_edge("tools","llm")

In [ ]:
app = workflow.compile()

In [ ]:
from IPython.display import Image,display
display(Image(app.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
result = app.invoke({
    "messages":[HumanMessage(content="What is the price of TCS.NS today?")]
    })

In [ ]:
for m in result['messages']:
    m.pretty_print()

In [ ]:
result = app.invoke({
    "messages":[HumanMessage(content="what was the latest union budget in india?")]
    })

In [ ]:
for m in result['messages']:
    m.pretty_print()

In [ ]:
result = app.invoke({
    "messages":[HumanMessage(content="what is the values of multipying 3590 with 7897? ")]
    })

In [ ]:
for m in result['messages']:
    m.pretty_print()

In [ ]:
result = app.invoke({
    "messages":[HumanMessage(content="give me latest news about AI and count the length of the response and multipy the response with 10?")]
    })

In [ ]:
for m in result['messages']:
    m.pretty_print()